---
# `Messages, Message Types, Dynamic Multi-Message System & ChatPromptTemplate in LangChain`
---

## 1. What are Messages in LangChain?

In a chat-based LLM application, we generally don't send only one plain string. Instead, we send a **sequence of messages**, where each message has a specific role and content.

A message can be represented conceptually as:

```text
Message = Role + Content
```

For example:

```text
System Message → You are a helpful AI assistant.
Human Message  → Explain RAG.
AI Message     → RAG stands for Retrieval-Augmented Generation...
```

This structure allows the model to understand **who is saying what** and what instructions should have higher priority.

---

# 2. Why Do We Need Different Message Types?

Consider a chatbot:

```text
System → You are an expert Python teacher.

Human → Explain decorators.

AI → A decorator is a function that modifies...

Human → Give me an example.

AI → Here's a simple example...
```

Each message has a different purpose.

* **System** → Defines behavior/instructions
* **Human** → User's input
* **AI** → Model's previous response
* **Tool** → Result returned from an external tool

This becomes especially important when building:

* Chatbots
* RAG applications
* AI agents
* Multi-turn conversations
* Tool-calling applications
* Customer-support systems

---

# 3. Types of Messages in LangChain

The commonly used message types include:

```text
SystemMessage
HumanMessage
AIMessage
ToolMessage
```

LangChain also provides message abstractions such as `BaseMessage`.

---

## 3.1 SystemMessage

A **SystemMessage** provides instructions or context that guide the behavior of the AI model.

Example:

```python
from langchain_core.messages import SystemMessage

message = SystemMessage(
    content="You are an expert LangChain mentor."
)
```

The model should follow this instruction when generating its response.

### Example

```text
System:
You are a helpful Python teacher.

Human:
Explain decorators.
```

### Use Cases

* Define AI's role
* Set behavior
* Provide rules
* Specify response style
* Give application-level instructions

### Interview Point

> `SystemMessage` is primarily used to define the behavior, role, and instructions for the AI model.

---

# 3.2 HumanMessage

A **HumanMessage** represents input from the user.

```python
from langchain_core.messages import HumanMessage

message = HumanMessage(
    content="Explain RAG in simple terms."
)
```

Example:

```text
Human:
Explain RAG.
```

### Use Cases

* User questions
* User instructions
* Chat input
* Dynamic application input

---

# 3.3 AIMessage

An **AIMessage** represents a response generated by the AI model.

For example:

```text
Human:
What is RAG?

AI:
RAG stands for Retrieval-Augmented Generation.
```

The second message is an `AIMessage`.

You can create one manually:

```python
from langchain_core.messages import AIMessage

message = AIMessage(
    content="RAG stands for Retrieval-Augmented Generation."
)
```

But normally, you don't manually create the AI response. The chat model generates it.

```python
response = model.invoke(
    "Explain RAG."
)

print(response)
```

The response is typically an `AIMessage`.

---

# 3.4 ToolMessage

A **ToolMessage** contains the result returned by a tool after an AI model requests that tool.

This becomes important in **AI Agents and tool calling**.

Example flow:

```text
Human
  ↓
AI decides to call calculator
  ↓
Tool executes
  ↓
ToolMessage
  ↓
AI generates final answer
```

Conceptually:

```text
Human:
What is 25 × 40?

AI:
Call calculator.

Tool:
1000

AI:
The answer is 1000.
```

The tool's result is represented using a `ToolMessage`.

---

# 4. Message Hierarchy

A simplified view:

```text
BaseMessage
│
├── SystemMessage
├── HumanMessage
├── AIMessage
└── ToolMessage
```

You don't always need to manually create these classes. LangChain's prompt templates can generate the appropriate messages for you.

---

# 5. Working with Multiple Messages

We can pass multiple messages to a chat model.

```python
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    SystemMessage,
    HumanMessage
)

model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

messages = [
    SystemMessage(
        content="You are an expert LangChain teacher."
    ),
    HumanMessage(
        content="Explain RAG."
    )
]

response = model.invoke(messages)

print(response.content)
```

### Flow

```text
SystemMessage
      +
HumanMessage
      ↓
   Chat Model
      ↓
  AIMessage
```

---

# 6. What is a Multi-Message System?

A **multi-message system** means sending multiple messages to the chat model instead of a single text prompt.

For example:

```text
System Message
      ↓
Human Message
      ↓
AI Message
      ↓
Human Message
      ↓
AI Message
```

This is the foundation of **multi-turn conversations**.

---

# 7. Why Dynamic Multi-Message Systems?

In real applications, messages are usually **dynamic**.

For example, suppose we are building a coding mentor application.

The conversation could be:

```text
System:
You are an expert Python teacher.

Human:
What is a decorator?

AI:
A decorator is...

Human:
Give me an example.

AI:
Here is an example...

Human:
Explain that example step-by-step.
```

We don't want to manually construct this entire conversation every time.

Instead, we can dynamically create the messages.

---

# 8. Dynamic Multi-Message System

A dynamic multi-message system contains:

```text
Static Messages
+
Dynamic Messages
+
Conversation History
```

For example:

```text
System Message
      ↓
Conversation History
      ↓
Current User Question
      ↓
Chat Model
```

The conversation history itself can contain:

```text
HumanMessage
AIMessage
HumanMessage
AIMessage
...
```

---

# 9. Example Without Prompt Template

We can manually create messages:

```python
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

messages = [
    SystemMessage(
        content="You are a helpful Python teacher."
    ),

    HumanMessage(
        content="What is a decorator?"
    ),

    AIMessage(
        content="A decorator is a function that modifies another function."
    ),

    HumanMessage(
        content="Give me a simple example."
    )
]
```

Then:

```python
response = model.invoke(messages)

print(response.content)
```

This works, but it becomes difficult to manage when the messages become dynamic.

That's where **ChatPromptTemplate** becomes useful.

---

# 10. What is ChatPromptTemplate?

`ChatPromptTemplate` is a LangChain class used to create **structured, reusable chat prompts containing multiple messages**.

Instead of creating every message manually, we define a template.

Example:

```python
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {domain} teacher."),
    ("human", "Explain {topic} in simple terms.")
])
```

Here we have two messages:

```text
System → You are an expert {domain} teacher.
Human  → Explain {topic} in simple terms.
```

Both `{domain}` and `{topic}` are dynamic variables.

---

# 11. Using ChatPromptTemplate

```python
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {domain} teacher."),
    ("human", "Explain {topic} in simple terms.")
])

messages = prompt.invoke({
    "domain": "LangChain",
    "topic": "RAG"
})

print(messages)
```

The template dynamically generates:

```text
System:
You are an expert LangChain teacher.

Human:
Explain RAG in simple terms.
```

---

# 12. ChatPromptTemplate + Chat Model

This is where things become powerful.

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert {domain} teacher."
    ),
    (
        "human",
        "Explain {topic} in simple terms."
    )
])

chain = prompt | model

response = chain.invoke({
    "domain": "LangChain",
    "topic": "RAG"
})

print(response.content)
```

### Flow

```text
User Inputs
    │
    ├── domain = LangChain
    └── topic  = RAG
          ↓
ChatPromptTemplate
          ↓
SystemMessage
          ↓
HumanMessage
          ↓
ChatOpenAI
          ↓
AIMessage
          ↓
response.content
```

---

# 13. Multiple Dynamic Messages

We can have multiple dynamic messages.

```python
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert {role}."
    ),
    (
        "human",
        "Explain {topic}."
    ),
    (
        "human",
        "Use a {style} explanation."
    )
])
```

Invoke:

```python
response = prompt.invoke({
    "role": "GenAI mentor",
    "topic": "Vector Database",
    "style": "beginner-friendly"
})
```

The generated messages become:

```text
System:
You are an expert GenAI mentor.

Human:
Explain Vector Database.

Human:
Use a beginner-friendly explanation.
```

---

# 14. Dynamic Conversation History

One of the most important applications is inserting **conversation history dynamically**.

Suppose we have:

```python
history = [
    ("human", "What is RAG?"),
    ("ai", "RAG stands for Retrieval-Augmented Generation.")
]
```

We can use `MessagesPlaceholder`.

```python
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful GenAI mentor."
    ),

    MessagesPlaceholder(
        variable_name="history"
    ),

    (
        "human",
        "{question}"
    )
])
```

Now invoke:

```python
response = chain.invoke({
    "history": history,
    "question": "Why is it useful?"
})
```

Conceptually, LangChain creates:

```text
System:
You are a helpful GenAI mentor.

Human:
What is RAG?

AI:
RAG stands for Retrieval-Augmented Generation.

Human:
Why is it useful?
```

This is extremely useful for conversational applications.

---

# 15. Why `MessagesPlaceholder`?

`MessagesPlaceholder` allows us to dynamically insert a **list of messages** into a prompt.

For example:

```python
MessagesPlaceholder(
    variable_name="history"
)
```

means:

> "At this position, insert whatever messages are provided through `history`."

This is different from:

```python
"{history}"
```

because `history` is expected to be a **collection of actual chat messages**, not just a string.

---

# 16. Complete Dynamic Chat Application

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert GenAI mentor."
    ),

    MessagesPlaceholder(
        variable_name="history"
    ),

    (
        "human",
        "{question}"
    )
])

chain = prompt | model

history = []

while True:

    question = input("You: ")

    if question.lower() == "exit":
        break

    response = chain.invoke({
        "history": history,
        "question": question
    })

    print("AI:", response.content)

    history.append(("human", question))
    history.append(("ai", response.content))
```

### Application Architecture

```text
                User
                 │
                 ↓
            User Question
                 │
                 ↓
       ┌──────────────────┐
       │ Conversation     │
       │ History          │
       └────────┬─────────┘
                │
                ↓
      ChatPromptTemplate
                │
        ┌───────┴────────┐
        ↓                ↓
 System Message    Message History
        │                │
        └───────┬────────┘
                ↓
          Current Question
                │
                ↓
          OpenAI Chat Model
                │
                ↓
            AIMessage
                │
                ↓
             User
```

---

# 17. Different Ways to Create ChatPromptTemplate

## Method 1: `from_messages()`

Most useful when working with multiple messages.

```python
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}")
])
```

---

## Method 2: `from_template()`

Useful for a simple single-message prompt.

```python
prompt = ChatPromptTemplate.from_template(
    "Explain {topic}."
)
```

This is essentially a simple prompt structure.

---

# 18. Message History vs Prompt Template

These concepts are related but different.

### Prompt Template

Defines **how the prompt should be structured**.

```text
System
+
History
+
Current Question
```

### Message History

Contains the **actual previous conversation**.

```text
Human → What is RAG?
AI    → RAG is...
Human → Why is it useful?
```

Together:

```text
ChatPromptTemplate
        +
Message History
        +
Current Input
        ↓
      LLM
```

---

# 19. Important Interview Distinction

### `PromptTemplate`

Generally used for text/string-based prompts.

```python
from langchain_core.prompts import PromptTemplate
```

Example:

```python
prompt = PromptTemplate.from_template(
    "Explain {topic}."
)
```

### `ChatPromptTemplate`

Designed for **chat models and multiple message roles**.

```python
from langchain_core.prompts import ChatPromptTemplate
```

Example:

```python
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert."),
    ("human", "{question}")
])
```

### Easy Way to Remember

```text
PromptTemplate
      ↓
String Prompt

ChatPromptTemplate
      ↓
List of Chat Messages
      ↓
Chat Model
```

---

# 20. Message Types vs Roles

A common interview confusion is between **message classes** and **role labels**.

In LangChain, you may see:

```python
("system", "...")
("human", "...")
("ai", "...")
```

These correspond conceptually to:

```python
SystemMessage(...)
HumanMessage(...)
AIMessage(...)
```

For example:

```python
("system", "You are helpful.")
```

is a convenient way of defining a system message inside `ChatPromptTemplate`.

---

# 21. Interview Questions & Answers

## Beginner

### 1. What is a message in LangChain?

A message represents one unit of communication between the system, user, AI, or tool. It contains a role/type and content.

---

### 2. What are the major message types?

Common types are:

* `SystemMessage`
* `HumanMessage`
* `AIMessage`
* `ToolMessage`

---

### 3. What is `SystemMessage`?

It provides instructions that define the AI model's behavior, role, or constraints.

---

### 4. What is `HumanMessage`?

It represents input provided by the user.

---

### 5. What is `AIMessage`?

It represents a response generated by the AI model.

---

### 6. What is `ToolMessage`?

It represents the result returned by a tool after the AI requests a tool call.

---

## Intermediate

### 7. What is `ChatPromptTemplate`?

`ChatPromptTemplate` is used to create reusable, structured prompts containing one or more chat messages and dynamic variables.

---

### 8. Why use `from_messages()`?

It allows us to define a sequence of messages with different roles.

```python
ChatPromptTemplate.from_messages([
    ("system", "..."),
    ("human", "{question}")
])
```

---

### 9. What is `MessagesPlaceholder`?

It is used to dynamically insert a list of messages, such as conversation history, into a chat prompt.

---

### 10. Why can't we simply use `{history}` instead of `MessagesPlaceholder`?

Because conversation history is typically a **list of structured messages**, not simply a string.

`MessagesPlaceholder` tells LangChain to insert those messages into the prompt as individual chat messages.

---

## Scenario-Based

### 11. How would you maintain conversation context in a chatbot?

I would maintain a list of previous messages and dynamically insert it into a `ChatPromptTemplate` using `MessagesPlaceholder`.

```text
System Message
      ↓
Conversation History
      ↓
Current User Message
      ↓
Chat Model
```

---

### 12. How would you create a chatbot where the system instruction remains fixed but user questions change?

Use a `ChatPromptTemplate`:

```python
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}")
])
```

Then pass different `question` values during invocation.

---

### 13. How would you add previous conversation dynamically?

Use:

```python
MessagesPlaceholder(
    variable_name="history"
)
```

and pass the history during invocation.

---

### 14. How would you design a customer-support chatbot using messages?

A typical structure would be:

```text
SystemMessage
→ Defines customer-support behavior

Message History
→ Maintains previous conversation

HumanMessage
→ Current customer question

Chat Model
→ Generates AIMessage
```

---

# Key Takeaways

* A **message** represents a unit of communication in a chat application.
* Common LangChain message types:

  * `SystemMessage`
  * `HumanMessage`
  * `AIMessage`
  * `ToolMessage`
* `SystemMessage` defines the AI's behavior and instructions.
* `HumanMessage` represents user input.
* `AIMessage` represents model-generated output.
* `ToolMessage` represents tool execution results.
* **Multi-message systems** allow multiple messages to be sent together.
* **Dynamic multi-message systems** allow message content/history to change at runtime.
* `ChatPromptTemplate` creates reusable prompts containing multiple messages.
* `from_messages()` is used to define a sequence of chat messages.
* `MessagesPlaceholder` is particularly useful for dynamically inserting conversation history.
* A typical conversational LangChain architecture is:

```text
User Input
    ↓
Message History
    ↓
ChatPromptTemplate
    ↓
Chat Model
    ↓
AIMessage
    ↓
Update History
    ↓
Next User Input
```

### ⭐ Interview One-Liner

> **`ChatPromptTemplate` allows us to define a reusable sequence of system, human, AI, and dynamic history messages, which can then be passed to a chat model to generate context-aware responses.**
